In [ ]:
import time
import yfinance as yf
import pandas as pd
from pathlib import Path
import random
import os
from FinMind.data import DataLoader
from google.colab import userdata
from datetime import datetime, timedelta
import numpy as np
import logging
import sys
from zoneinfo import ZoneInfo
import logging
import io
import atexit
# !pip install FinMind
# 上市首日收盤
# 上市首日最高
# 上市首日最低
# 最低投標價格(元)
# 最低得標價格(元)
# 得標加權平均價格(元)



In [ ]:
def init_output_logging(folder_path, script_name, timezone="Asia/Taipei"):
    log_buffer = io.StringIO()
    logger = logging.getLogger(script_name) # 使用專屬名稱避免衝突
    logger.setLevel(logging.ERROR)

    # 清理舊的 Handler
    for handler in logger.handlers[:]:
        logger.removeHandler(handler)

    formatter = logging.Formatter('%(asctime)s [%(levelname)s]: %(message)s', '%Y-%m-%d %H:%M:%S')
    stream_handler = logging.StreamHandler(log_buffer)
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)

    def save_action():
        content = log_buffer.getvalue()
        if content:
            log_dir = os.path.join(folder_path, "logs")
            os.makedirs(log_dir, exist_ok=True)
            timestamp = datetime.now(ZoneInfo(timezone)).strftime("%Y%m%d_%H%M%S")
            final_path = os.path.join(log_dir, f"{script_name}_{timestamp}.log")

            with open(final_path, "w", encoding="utf-8") as f:
                f.write(content)
            print(f"\n⚠️ 錯誤已統一儲存至: {final_path}")
            log_buffer.truncate(0) # 存完清空，避免重複存檔
            log_buffer.seek(0)
        else:
            print("\n✅ 執行結束，無錯誤紀錄。")

    # 綁定 save 方法到 logger 物件上，方便手動呼叫
    logger.save = save_action

    # 依然註冊 atexit 作為保險
    atexit.register(save_action)

    # 攔截全域崩潰
    def handle_unhandled_exception(exc_type, exc_value, exc_traceback):
        if issubclass(exc_type, KeyboardInterrupt):
            sys.__excepthook__(exc_type, exc_value, exc_traceback)
            return
        logger.error("未捕捉的致命錯誤", exc_info=(exc_type, exc_value, exc_traceback))

    sys.excepthook = handle_unhandled_exception

    return logger


In [ ]:

def get_Target_value(api, code:str, datetime:pd.Timestamp, feature_cols:list):
    """
    得到完整上市當天股價
    """
    date_str = datetime.strftime('%Y-%m-%d')
    df = api.taiwan_stock_daily(stock_id=code,start_date=date_str,end_date=date_str)
    df = df.iloc[:1, 2:]
    if df is None or df.empty:
        return None
    df.columns = feature_cols[:8]
    return df.round(3)

# api = DataLoader()
# feature_cols = ["成交量", "成交金額", "開盤價", "最高價", "最低價", "收盤價", "價差", "成交筆數"]
# df = get_Target_value(api, "6026", pd.Timestamp("2016-01-27"), feature_cols)
# df

In [ ]:
# 1. 讀取當前要抓取的類型的儲存表沒有則創建
# 2. 產生與競拍資訊標中新增的股票代號跟投標開始日
def search_index_list(raw_data_path, curr_data_path, code_col:str, time_col:list, feature_cols:list):
    """
    curr_data: 上市櫃行情表, pd.DataFrame
    diff_index: 與原始競拍資料差異, pd.Index
    """
    if not os.path.exists(raw_data_path):
        return
    raw_data = pd.read_csv(raw_data_path, encoding="utf-8-sig", dtype={code_col:str})
    raw_data[time_col] = raw_data[time_col].apply(pd.to_datetime, format='mixed')
    if os.path.exists(curr_data_path):
        curr_data = pd.read_csv(curr_data_path, encoding="utf-8-sig", dtype={code_col:str})
        curr_data[time_col] = curr_data[time_col].apply(pd.to_datetime, format='mixed')
    else:
        curr_data = pd.DataFrame(columns=[code_col]+time_col+feature_cols)

    raw_indexed = raw_data.set_index([code_col, time_col[0], time_col[1]]) # 都將第一位放入投標開始日

    if not curr_data.empty:
        curr_indexed = curr_data.set_index([code_col, time_col[0], time_col[1]])
        diff_index = raw_indexed.index.difference(curr_indexed.index)
    else:
        diff_index = raw_indexed.index

    return curr_data, diff_index, raw_data

In [ ]:
# 最低投標價格： bid_min
# 最低得標價格： win_min
# 得標加權平均價格： win_avg
# "首日收盤溢價率", "首日最高溢價率", "首日最低溢價率", "最低得標加價率", "加權平均加價率"

# 計算之後 y 的特徵值
def cal_y_feature(get_data, raw_data, code:str, bid_date:pd.Timestamp):
    """
    1. 門檻預測 : 最低得標加價率 => (最低得標價格 / 最低投標價格) - 1
    2. 行情預測 : 加權平均加價率 => (得標加權平均價 / 最低投標價格) - 1
    3. 獲利預估 : 預估獲利率 => (首日收盤價 / 最低投標價格) - 1
    """
    prices = [raw_data.loc[(raw_data["證券代號"] == code) & (raw_data["投標開始日"] == bid_date), col].squeeze()
        for col in ["最低投標價格(元)", "最低得標價格(元)", "得標加權平均價格(元)"]]
    bid_min, win_min, win_avg = [p if not isinstance(p, pd.Series) else np.nan for p in prices]

    close_p = get_data["收盤價"].iloc[0]

    get_data["預估獲利率"] = (close_p / bid_min) - 1
    get_data["最低得標加價率"] = (win_min / bid_min) - 1
    get_data["加權平均加價率"] = (win_avg / bid_min) - 1

    return get_data.round(3)





In [ ]:
def main():
    folder = Path("/content/drive/MyDrive/Colab Notebooks/stock_auction_pred_project")
    result_folder = "csv"
    raw_data_path = folder/ result_folder / "bid_info.csv"
    target_variable_path = folder/ result_folder / "target_variable.csv"
    code_col = "證券代號"
    time_col = ["投標開始日", "撥券日期(上市、上櫃日期)"]
    feature_cols = ["成交量", "成交金額", "開盤價", "最高價", "最低價", "收盤價", "價差", "成交筆數",
                    "首日收盤溢價率", "首日最高溢價率", "首日最低溢價率", "最低得標加價率", "加權平均加價率"]

    start = time.time()
    curr_data, diff_index, raw_data = search_index_list(raw_data_path, target_variable_path, code_col, time_col, feature_cols)

    dl = DataLoader()
    token = userdata.get('finmind2')
    dl.login_by_token(api_token=token)

    MAX_CALLS_PER_HOUR = 595  # 每小時上限，給5個token的緩衝
    call_count = 0
    current_hour = datetime.now().hour
    dflist = []
    my_logger = init_output_logging(folder, "目標變數")

    try:
        for code, bid_date, list_date in diff_index:
            now = datetime.now() # 此次迴圈是哪一個小時
            if now.hour != current_hour:
                current_hour = now.hour
                call_count = 0
                print(f"API 次數已重置")
            if call_count >= MAX_CALLS_PER_HOUR:
                # 計算距離下一個整點還有幾秒
                next_hour = (now + timedelta(hours=1)).replace(minute=0, second=5, microsecond=0)
                wait_seconds = (next_hour - now).total_seconds()
                print(f"已達上限 {MAX_CALLS_PER_HOUR} 次，休息 {int(wait_seconds)} 秒至下個整點...")
                time.sleep(wait_seconds)

                # 休息完後更新狀態
                current_hour = datetime.now().hour
                call_count = 0
            call_count += 1
            try:
                print(f"股票代號: {code}, 投標開始日期: {bid_date}, 上市日期: {list_date}")
                data = get_Target_value(dl, code, list_date, feature_cols)
                if data is None:
                    print("===> 無資料")
                    continue
                data = cal_y_feature(data, raw_data, code, bid_date)
                data.insert(0, "撥券日期(上市、上櫃日期)", list_date)
                data.insert(0, "投標開始日", bid_date)
                data.insert(0, "證券代號", code)

                data_dict = data.to_dict(orient='records')[0]
                print(data_dict)

                dflist.append(data)
                time.sleep(random.uniform(1, 2))
            except Exception as e:
                my_logger.exception("錯誤")
                print(e)

    except Exception as e:
        print(e)

    finally:
        my_logger.save()
        if dflist:
            new_df = pd.concat(dflist)
            if curr_data.empty:
                curr_data = new_df
            else:
                curr_data = pd.concat([curr_data, new_df])
            curr_data[feature_cols] = curr_data[feature_cols].apply(pd.to_numeric, errors='coerce')
            curr_data.to_csv(target_variable_path, index=False, encoding="utf-8-sig")
        else:
            print("**此次無資料**")

    end = time.time()
    print(f"**完成** -->花費時間: {end-start} 秒")

if __name__ == "__main__":
    main()

In [ ]:
# raw_data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/stock_auction_pred_project/csv/bid_info.csv")
# curr_data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/stock_auction_pred_project/csv/target_variable.csv")
# raw_data_path = "/content/drive/MyDrive/Colab Notebooks/stock_auction_pred_project/csv/bid_info.csv"
# curr_data_path = "/content/drive/MyDrive/Colab Notebooks/stock_auction_pred_project/csv/target_variable.csv"
# print(len(curr_data))
# print(raw_data.columns)
# code_col = "證券代號"
# time_col = ["投標開始日", "撥券日期(上市、上櫃日期)"]
# feature_cols = ["成交量", "成交金額", "開盤價", "最高價", "最低價", "收盤價", "價差", "成交筆數",
#                 "首日收盤溢價率", "首日最高溢價率", "首日最低溢價率", "最低得標加價率", "加權平均加價率"]
# c, i, r = search_index_list(raw_data_path, curr_data_path, code_col, time_col, feature_cols)
# print(i)

434
Index(['開標日期', '證券名稱', '證券代號', '發行市場', '發行性質', '競拍方式', '投標開始日', '投標結束日',
       '競拍數量(張)', '最低投標價格(元)', '最低每標單投標數量(張)', '最高投(得)標數量(張)', '保證金成數(%)',
       '每一投標單投標處理費(元)', '撥券日期(上市、上櫃日期)', '主辦券商', '得標總金額(元)', '得標手續費率(%)',
       '總合格件', '合格投標數量(張)', '最低得標價格(元)', '最高得標價格(元)', '得標加權平均價格(元)', '承銷價格(元)',
       '取消競價拍賣(流標或取消)', 'update_time'],
      dtype='object')
